In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from adjustText import adjust_text
import os
import pickle

from pydeseq2.dds import DeseqDataSet
from pydeseq2.default_inference import DefaultInference
from pydeseq2.ds import DeseqStats
from pydeseq2.utils import load_example_data

from collections import Counter
from upsetplot import UpSet
from scipy import stats
import gseapy as gp
from gseapy import barplot, dotplot

import matplotlib
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42

import run_pydeseq2_plots as funcs
import time as time

In [3]:
GTEx_dir = "../../../../GTEx_bulk/"

In [4]:
count_matrix = pd.read_csv(GTEx_dir + "01_count_matrix.csv", index_col = 0)
metadata = pd.read_csv(GTEx_dir + "01_metadata.csv", index_col = 0)

### Downsample to the same number of donors as snRNA-seq

In [12]:
np.random.seed(42)  # for reproducibility
sampled_donors = np.random.choice(metadata.index, size=299, replace=False)

# subset both metadata and count matrix
metadata_sub = metadata.loc[sampled_donors]
count_matrix_sub = count_matrix.loc[:, metadata_sub.SAMPID]

In [15]:
transposed_count_matrix = count_matrix_sub.T
metadata_sub.index = metadata_sub.SAMPID

In [16]:
contrasts = [
        ("age_group", "old", "young"),
        ("sex", "male", "female"),
    ]

In [18]:
results_dict, significant_genes, dds = funcs.run_deseq_analysis(transposed_count_matrix,
                                                                    metadata_sub, contrasts,
                                                                    covariate_keys = ["sex", "age_group"])

Fitting size factors...


Using None as control genes, passed at DeseqDataSet initialization


... done in 0.66 seconds.

Fitting dispersions...
... done in 9.15 seconds.

Fitting dispersion trend curve...
... done in 1.15 seconds.

Fitting MAP dispersions...
... done in 10.39 seconds.

Fitting LFCs...
... done in 7.45 seconds.

Calculating cook's distance...
... done in 1.24 seconds.

Replacing 757 outlier genes.

Fitting dispersions...
... done in 0.21 seconds.

Fitting MAP dispersions...
... done in 0.20 seconds.

Fitting LFCs...
... done in 0.18 seconds.

Running Wald tests...
... done in 3.07 seconds.



Log2 fold change & Wald test p-value: age_group old vs young
                 baseMean  log2FoldChange     lfcSE      stat    pvalue  \
DDX11L1      6.396868e-01       -0.341412  0.547132 -0.624002  0.532626   
WASH7P       5.902850e+01       -0.017519  0.167258 -0.104743  0.916580   
MIR6859-1    2.374226e-02        0.390795  3.394236  0.115135  0.908338   
MIR1302-2HG  1.317157e+00        0.194439  0.522557  0.372092  0.709824   
FAM138A      6.516240e-01        0.106987  0.580881  0.184181  0.853872   
...                   ...             ...       ...       ...       ...   
MT-ND6       3.063756e+05        0.327595  0.236904  1.382820  0.166720   
MT-TE        1.437445e+02        0.387257  0.330036  1.173376  0.240645   
MT-CYB       2.887440e+06        0.694492  0.234583  2.960541  0.003071   
MT-TT        7.271997e+00        0.922012  0.359419  2.565287  0.010309   
MT-TP        1.984271e+01        1.082545  0.372472  2.906383  0.003656   

                 padj  
DDX11L1       

Running Wald tests...
... done in 2.89 seconds.



Log2 fold change & Wald test p-value: sex male vs female
                 baseMean  log2FoldChange     lfcSE      stat    pvalue  \
DDX11L1      6.396868e-01       -0.487474  0.318186 -1.532038  0.125513   
WASH7P       5.902850e+01        0.198575  0.097303  2.040783  0.041272   
MIR6859-1    2.374226e-02       -0.055044  1.960209 -0.028081  0.977598   
MIR1302-2HG  1.317157e+00       -0.285873  0.303215 -0.942808  0.345779   
FAM138A      6.516240e-01       -0.585030  0.324614 -1.802231  0.071509   
...                   ...             ...       ...       ...       ...   
MT-ND6       3.063756e+05       -0.061381  0.136887 -0.448403  0.653863   
MT-TE        1.437445e+02       -0.017547  0.190731 -0.091996  0.926701   
MT-CYB       2.887440e+06        0.165256  0.135546  1.219187  0.222773   
MT-TT        7.271997e+00        0.192078  0.207754  0.924547  0.355201   
MT-TP        1.984271e+01        0.344197  0.214673  1.603353  0.108857   

                 padj  
DDX11L1           

In [19]:
results_dict.keys()

dict_keys(['age_group_old_vs_young', 'sex_male_vs_female'])

In [20]:
results_dict['age_group_old_vs_young'].to_csv("01_GTEx_subsampled_bulk_age_group_old_vs_young.csv")

In [21]:
results_dict['sex_male_vs_female'].to_csv("01_GTEx_subsampled_sex_male_vs_female_DESeq2.csv")